# 03. Published methods, weights-only and as published
Each published method runs from its authors' code under our validations (`DECISIONS.md`), twice: as published, with weights and activations as the papers show, and weights-only, with activations at full precision. This notebook places each method on the accuracy–size chart and compares it with its paper.

With `RERUN = False` it reads the saved results in `outputs/`. With `RERUN = True` it calls each method's `run.sh`, which takes from minutes to two hours per point. That needs the authors' code, set up by `scripts/setup_third_party.sh`, and their environment in `ADALOG_PY` (see the README).

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

import glob
import os
import subprocess
import sys

REPO = Path.cwd()
THIRD = Path(os.environ.get("THIRD_PARTY", REPO.parent / "third_party"))   # scripts/setup_third_party.sh
PY = sys.executable                                     # this environment runs our scripts
os.environ.setdefault("PY", PY)
os.environ.setdefault("ADALOG_PY", "python")            # the published methods' environment, see the README

def sh(cmd):
    """Run a command from the repository root, show its output as it comes, and stop if it fails."""
    with subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as p:
        for line in p.stdout:
            print(line, end="")
    if p.returncode:
        raise RuntimeError(f"exit code {p.returncode}: {cmd}")

def latest(pattern):
    """The newest file under ../third_party that matches `pattern`."""
    found = sorted(glob.glob(str(THIRD / pattern)), key=os.path.getmtime)
    if not found:
        raise FileNotFoundError(pattern)
    return found[-1]

def baselines():
    """The dense baselines of notebook 02, as {model: {"size_mb": ..., "accuracy": ...}}."""
    b = json.loads((Path("outputs") / "baselines_summary.json").read_text())
    return {k: {"size_mb": b["size_mb"][k], "accuracy": a} for k, a in b["accuracy"].items()}

def checkpoint_of_run(name):
    """The checkpoint a FIMA-Q or APHQ-ViT run saved after block reconstruction, read from its log."""
    folder = THIRD / ("APHQ-ViT-protocol" if name.startswith("APHQ") else "FIMA-Q-protocol")
    log = (folder / f"run_{name.replace(' ', '_')}.log").read_text(errors="replace")
    saved = [l.split("Saving checkpoint to ")[1].strip() for l in log.splitlines() if "Saving checkpoint to" in l and "optimsize" in l]
    return str(folder / saved[-1])

def pack(method, ckpt, out, bits, as_published=False):
    """Pack a method's checkpoint into a real file, checked bit for bit against the method's own weights."""
    sh(f'"{PY}" scripts/quantize/pack_published.py --method {method} --bits {bits} --ckpt "{ckpt}" --out {out}'
       + (" --as-published" if as_published else ""))

def attach_file(name, out, ckpt):
    """A run as published is scored inside the method's forward; its entry gets the packed file, its size and checkpoint."""
    path = OUT / "published_summary.json"
    s = json.loads(path.read_text())
    s[name].update(file=out, size_mb=os.path.getsize(out) / 1e6, checkpoint=os.path.relpath(ckpt))
    path.write_text(json.dumps(s, indent=2, default=float))

RERUN = False
OUT = Path("outputs")
SETS = ["ImageNet-1k val", "ImageNet-V2", "ImageNet-R"]
pub = json.loads((OUT / "published_summary.json").read_text())

def top1(e, s="ImageNet-1k val"):
    return e["accuracy"][s]["top1"]

base = baselines()
ti_int8, s_fp32 = base["DeiT-Ti INT8"], base["DeiT-S FP32"]
print("DeiT-Ti INT8:", round(ti_int8["size_mb"], 2), "MB", round(top1(ti_int8), 2))
print("DeiT-S FP32:", round(s_fp32["size_mb"], 2), "MB", round(top1(s_fp32), 2))

Paper values on DeiT-S, ImageNet val, checked against the local PDFs or the logs. AdaLog's are for its as-published runs.

In [ ]:
PAPER = {
    "AdaLog W3A3": 24.47, "AdaLog W4A4": 72.06, "AdaLog W6A6": 79.39,
    "APHQ-ViT W4A4": 76.40, "APHQ-ViT W3A3": 68.76,
    "FIMA-Q W4A4": 76.87, "FIMA-Q W3A3": 69.13, "FIMA-Q W6A6": 79.52,
    "RepQ-ViT W4A4": 69.03, "RepQ-ViT W6A6": 78.90,
    "PTQ4ViT W8A8": 79.47, "PTQ4ViT W6A6": 76.28,
}
COLORS = {"AdaLog": "#4C72B0", "APHQ-ViT": "#DD8452", "FIMA-Q": "#55A868", "RepQ-ViT": "#C44E52", "PTQ4ViT": "#8172B3"}

def table(method):
    rows = [(k, v) for k, v in pub.items() if k.startswith(method + " ")]
    print(f"{'point':18s} {'run':13s} {'MB':>7s} " + " ".join(f"{s.split()[-1]:>6s}" for s in SETS) + "   paper")
    for k, v in sorted(rows, key=lambda r: r[1].get("size_mb") or 0):
        run = v.get("run", "weights-only")
        accs = " ".join(f"{top1(v, s):6.2f}" for s in SETS)
        print(f"{k:18s} {run:13s} {v.get('size_mb') or 0:7.2f} {accs}   {PAPER.get(k, '')}")

def chart(methods, title):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.axhline(top1(ti_int8), ls=":", c="grey", lw=1.2, label="DeiT-Ti INT8 (limit)")
    ax.axhline(top1(s_fp32), ls="--", c="black", lw=0.8, alpha=0.5, label="DeiT-S FP32")
    for m in methods:
        for run, marker, fill in (("weights-only", "o", True), ("as published", "s", False)):
            pts = sorted((v["size_mb"], top1(v)) for k, v in pub.items()
                         if k.startswith(m + " ") and v.get("run", "weights-only") == run and v.get("size_mb"))
            if not pts:
                continue
            x, y = zip(*pts)
            ax.plot(x, y, ls="-" if fill else ":", marker=marker, c=COLORS[m], alpha=0.9 if fill else 0.6,
                    mfc=COLORS[m] if fill else "none", label=f"{m}, {run}")
    ax.set(xlabel="file size (MB)", ylabel="top-1 on ImageNet val (%)", title=title)
    ax.legend(fontsize=7, loc="lower right"); ax.grid(alpha=0.3)
    plt.show()

## AdaLog

Patch `methods/adalog/weights_only.patch`, 32 calibration images. The authors' own calibration draw was also run once and moves W3A3 by about 3 points.

In [ ]:
if RERUN:
    sh("bash methods/adalog/run.sh")   # weights-only W3, W4 and W6: calibration, packed file, score from the file
    data = os.environ.get("IMAGENET_DIR", str(REPO.parent / "data" / "imagenet"))
    for b in (3, 4, 6):                 # as published, scored inside AdaLog's own forward pass
        sh(f'cd "{THIRD}/AdaLog-protocol" && "$ADALOG_PY" "{REPO}/methods/adalog/score_published.py" --config ./configs/{b}bit.py '
           f'--dataset "{data}" --calib-list "{REPO}/splits/calib_seed0.txt" --name "AdaLog W{b}A{b}"')
        ckpt = latest(f"AdaLog-protocol/checkpoints/quant_result/*/deit_small_w{b}_a{b}_*calibsize_32.pth")
        out = f"outputs/artifacts/adalog_w{b}a{b}.safetensors"
        pack("adalog", ckpt, out, b, as_published=True)
        attach_file(f"AdaLog W{b}A{b}", out, ckpt)
    pub = json.loads((OUT / 'published_summary.json').read_text())
table('AdaLog')
chart(['AdaLog'], 'Published methods so far')

## APHQ-ViT

Patch `methods/aphq/protocol.patch`. As published, the authors' pipeline first reconstructs each MLP with ReLU in place of GELU, which serves activation quantization, then calibrates and reconstructs each block. The weights-only runs keep that MLP reconstruction too, so their files are scored with ReLU in the MLPs. The earlier weights-only runs without it stay in the table as "no MLP reconstruction" and are left off the chart.

In [ ]:
if RERUN:
    for b in (4, 3):   # as published: MLP reconstruction, calibration, block reconstruction
        name = f"APHQ-ViT W{b}A{b}"
        sh(f'bash methods/aphq/run.sh {b}bit/best.py "{name}" --reconstruct-mlp')
        ckpt, out = checkpoint_of_run(name), f"outputs/artifacts/aphq_w{b}a{b}.safetensors"
        pack("aphq", ckpt, out, b, as_published=True)
        attach_file(name, out, ckpt)
    # the MLP reconstruction does not depend on the bit width, so the weights-only runs reuse the one the W4A4 run saved
    mr = Path(checkpoint_of_run("APHQ-ViT W4A4")).parent / "deit_small_reconstructed.pth"
    for b in (4, 3):   # weights-only, scored from the packed file with ReLU in the MLPs
        name = f"APHQ-ViT W{b} MR"
        sh(f'bash methods/aphq/run.sh {b}bit/w{b}a32.py "{name}" --reconstruct-mlp --load-reconstruct-checkpoint "{mr}"')
        out = f"outputs/artifacts/aphq_w{b}_mr.safetensors"
        pack("aphq", checkpoint_of_run(name), out, b)
        sh(f'"{PY}" scripts/quantize/evaluate_file.py {out} --name "APHQ-ViT W{b}" --mlp-act relu')
    pub = json.loads((OUT / 'published_summary.json').read_text())
table('APHQ-ViT')
chart(['AdaLog', 'APHQ-ViT'], 'Published methods so far')

## FIMA-Q

Patch `methods/fimaq/protocol.patch`, 128 calibration images and 1 024 for block reconstruction. W2 is not a published setting (W3 config with 2-bit weights).

In [ ]:
if RERUN:
    for b in (3, 4, 6, 8):   # as published
        name = f"FIMA-Q W{b}A{b}"
        sh(f'bash methods/fimaq/run.sh {b}bit/best.py "{name}"')
        ckpt, out = checkpoint_of_run(name), f"outputs/artifacts/fimaq_w{b}a{b}.safetensors"
        pack("fimaq", ckpt, out, b, as_published=True)
        attach_file(name, out, ckpt)
    for b in (2, 3, 4, 6):   # weights-only, scored from the packed file
        name = f"FIMA-Q W{b}"
        sh(f'bash methods/fimaq/run.sh {b}bit/w{b}a32.py "{name}"')
        out = f"outputs/artifacts/fimaq_w{b}.safetensors"
        pack("fimaq", checkpoint_of_run(name), out, b)
        sh(f'"{PY}" scripts/quantize/evaluate_file.py {out} --name "{name}"')
    pub = json.loads((OUT / 'published_summary.json').read_text())
table('FIMA-Q')
chart(['AdaLog', 'APHQ-ViT', 'FIMA-Q'], 'Published methods so far')

## RepQ-ViT

Protocol wrapper `methods/repq/protocol.py`, 32 calibration images.

In [ ]:
if RERUN:
    sh("bash methods/repq/run.sh")   # W4A4 and W6A6 as published, W4 and W6 weights-only scored from their files
    pub = json.loads((OUT / 'published_summary.json').read_text())
table('RepQ-ViT')
chart(['AdaLog', 'APHQ-ViT', 'FIMA-Q', 'RepQ-ViT'], 'Published methods so far')

## PTQ4ViT

Our protocol wrapper `methods/ptq4vit/protocol.py`, 32 calibration images, Hessian-guided search.

In [ ]:
if RERUN:
    sh("bash methods/ptq4vit/run.sh")   # W8A8 and W6A6 as published, W8 and W6 weights-only scored from their files
    pub = json.loads((OUT / 'published_summary.json').read_text())
table('PTQ4ViT')
chart(['AdaLog', 'APHQ-ViT', 'FIMA-Q', 'RepQ-ViT', 'PTQ4ViT'], 'Published methods so far')

## The INT8 references and the GemLite check

torchao W8A8 has a real INT8 kernel that can be run on our GPU architecture(`scripts/quantize/w8a8_torchao.py`). GemLite runs FIMA-Q's packed files through low-bit kernels, and its scores should match the packed files.

In [ ]:
if RERUN:
    sh(f'"{PY}" scripts/quantize/w8a8_torchao.py')   # DeiT-S and DeiT-Ti
    for fmt in ["W0 A16", "W8 A16", "W8 A8", "W4 A16", "W4 A8", "W2 A16", "W2 A8", "W3 A16", "W3 A8", "W6 A16", "W6 A8"]:
        sh(f'PYTHONPATH="{THIRD}/gemlite" "{PY}" scripts/measure/gemlite_test.py run "{fmt}" --score')
for f in ["w8a8_summary.json", "gemlite_summary.json"]:
    for k, v in json.loads((OUT / f).read_text()).items():
        if "accuracy" in v:
            size = f"{v['size_mb']:7.2f} MB" if v.get("size_mb") else "      -   "   # GemLite runs FIMA-Q's files, sized above
            print(f"{k:24s} {size}  " + " / ".join(f"{top1(v, s):.2f}" for s in SETS))

## Weights-only against as published

The distance between the two runs of one method at the same bit width shows how much of its loss comes from the activations.

In [ ]:
for k, v in pub.items():
    if v.get("run") == "as published":
        m, wa = k.rsplit(" ", 1)
        w = f"{m} {wa.split('A')[0]}"
        if w in pub:
            print(f"{k:16s} {top1(v):6.2f}   weights-only {top1(pub[w]):6.2f}   activations cost {top1(pub[w]) - top1(v):5.2f}")